In [1]:
# import

import sys
from pathlib import Path
from types import SimpleNamespace
from collections import namedtuple

import torch
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.training.losses import (
    label_smoothed_cross_entropy_loss,
    compute_mt_loss,
    maybe_compute_loss_from_outputs,
)

from src.training.optimizer import (
    build_optimizer,
    build_scheduler,
)

from src.training.evaluate import (
    normalize_text,
    compute_bleu,
    compute_chrf,
    compute_rouge,
    compute_mt_metrics,
    evaluate_model,
)

from src.training.trainer import (
    train,
    _move_to_device,
    _extract_labels,
    _compute_total_training_steps,
)

In [2]:
# test loss basic

batch_size = 2
seq_len = 4
vocab_size = 10
ignore_index = -100

logits = torch.randn(batch_size, seq_len, vocab_size, requires_grad=True)
labels = torch.randint(0, vocab_size, (batch_size, seq_len))

loss, ce_loss = label_smoothed_cross_entropy_loss(
    logits=logits,
    labels=labels,
    label_smoothing=0.0,
    ignore_index=ignore_index,
)

assert loss.dim() == 0
assert ce_loss.dim() == 0
assert torch.allclose(loss, ce_loss)

loss.backward()

assert logits.grad is not None

print("loss basic passed")

loss basic passed


In [3]:
# test loss with ignore index

logits = torch.randn(2, 5, 12, requires_grad=True)
labels = torch.randint(0, 12, (2, 5))
labels[0, 2] = -100
labels[1, 4] = -100

loss, ce_loss = label_smoothed_cross_entropy_loss(
    logits=logits,
    labels=labels,
    label_smoothing=0.1,
    ignore_index=-100,
)

assert loss.dim() == 0
assert ce_loss.dim() == 0
assert torch.isfinite(loss)
assert torch.isfinite(ce_loss)

loss.backward()

assert logits.grad is not None

print("loss ignore index passed")

loss ignore index passed


In [4]:
# test loss with ignore index

logits = torch.randn(2, 5, 12, requires_grad=True)
labels = torch.randint(0, 12, (2, 5))
labels[0, 2] = -100
labels[1, 4] = -100

loss, ce_loss = label_smoothed_cross_entropy_loss(
    logits=logits,
    labels=labels,
    label_smoothing=0.1,
    ignore_index=-100,
)

assert loss.dim() == 0
assert ce_loss.dim() == 0
assert torch.isfinite(loss)
assert torch.isfinite(ce_loss)

loss.backward()

assert logits.grad is not None

print("loss ignore index passed")

loss ignore index passed


In [5]:
# test loss with ignore index

logits = torch.randn(2, 5, 12, requires_grad=True)
labels = torch.randint(0, 12, (2, 5))
labels[0, 2] = -100
labels[1, 4] = -100

loss, ce_loss = label_smoothed_cross_entropy_loss(
    logits=logits,
    labels=labels,
    label_smoothing=0.1,
    ignore_index=-100,
)

assert loss.dim() == 0
assert ce_loss.dim() == 0
assert torch.isfinite(loss)
assert torch.isfinite(ce_loss)

loss.backward()

assert logits.grad is not None

print("loss ignore index passed")

loss ignore index passed


In [6]:
# test maybe compute loss from tensor outputs

logits = torch.randn(2, 4, 10, requires_grad=True)
labels = torch.randint(0, 10, (2, 4))

loss, logs = maybe_compute_loss_from_outputs(
    outputs=logits,
    labels=labels,
    label_smoothing=0.0,
    ignore_index=-100,
)

assert loss.dim() == 0
assert "train/loss_total" in logs

print("maybe compute loss tensor passed")

maybe compute loss tensor passed


In [7]:
# test maybe compute loss from dict outputs

logits = torch.randn(2, 4, 10, requires_grad=True)
labels = torch.randint(0, 10, (2, 4))

outputs = {"logits": logits}

loss, logs = maybe_compute_loss_from_outputs(
    outputs=outputs,
    labels=labels,
)

assert loss.dim() == 0
assert "train/loss_ce" in logs

print("maybe compute loss dict passed")

maybe compute loss dict passed


In [8]:
# test maybe compute loss from object outputs

Output = namedtuple("Output", ["logits", "loss"])

manual_loss = torch.tensor(2.5, requires_grad=True)
outputs = Output(logits=None, loss=manual_loss)

loss, logs = maybe_compute_loss_from_outputs(
    outputs=outputs,
    labels=None,
)

assert loss is manual_loss
assert logs["train/loss_total"] == 2.5

print("maybe compute loss object passed")

maybe compute loss object passed


In [9]:
# test optimizer

model = torch.nn.Sequential(
    torch.nn.Linear(8, 16),
    torch.nn.LayerNorm(16),
    torch.nn.Linear(16, 5),
)

config = {
    "learning_rate": 1e-3,
    "weight_decay": 0.01,
    "group_weight_decay": True,
}

optimizer = build_optimizer(model, config)

assert isinstance(optimizer, torch.optim.AdamW)
assert len(optimizer.param_groups) == 2
assert optimizer.param_groups[0]["lr"] == 1e-3
assert optimizer.param_groups[0]["weight_decay"] == 0.01
assert optimizer.param_groups[1]["weight_decay"] == 0.0

print("optimizer passed")

optimizer passed


In [10]:
# test optimizer without grouped weight decay

model = torch.nn.Linear(8, 4)

config = {
    "lr": 5e-4,
    "weight_decay": 0.02,
    "group_weight_decay": False,
}

optimizer = build_optimizer(model, config)

assert isinstance(optimizer, torch.optim.AdamW)
assert len(optimizer.param_groups) == 1
assert optimizer.param_groups[0]["lr"] == 5e-4
assert optimizer.param_groups[0]["weight_decay"] == 0.02

print("optimizer no group passed")

optimizer no group passed


In [11]:
# test scheduler linear

model = torch.nn.Linear(8, 4)

optimizer = build_optimizer(
    model,
    {
        "learning_rate": 1e-3,
        "weight_decay": 0.0,
        "group_weight_decay": False,
    },
)

scheduler = build_scheduler(
    optimizer,
    {
        "scheduler_type": "linear",
        "warmup_steps": 2,
    },
    num_training_steps=6,
)

assert scheduler is not None

lrs = []
for _ in range(6):
    optimizer.step()
    scheduler.step()
    lrs.append(optimizer.param_groups[0]["lr"])

assert all(lr >= 0 for lr in lrs)

print("scheduler linear passed")

scheduler linear passed


In [12]:
# test scheduler none

model = torch.nn.Linear(8, 4)
optimizer = build_optimizer(model, {"learning_rate": 1e-3})

scheduler = build_scheduler(
    optimizer,
    {"scheduler_type": "none"},
    num_training_steps=10,
)

assert scheduler is None

print("scheduler none passed")

scheduler none passed


In [13]:
# test normalize text

text = "  Tôi   thích   dịch   máy  "
normalized = normalize_text(text)

assert normalized == "Tôi thích dịch máy"
assert normalize_text(None) == ""

print("normalize text passed")

normalize text passed


In [14]:
# test metric functions

predictions = [
    "tôi thích dịch máy",
    "bạn là học sinh",
]

references = [
    "tôi thích dịch máy",
    "bạn là học sinh",
]

try:
    bleu = compute_bleu(predictions, references)
    chrf = compute_chrf(predictions, references)
    rouge = compute_rouge(predictions, references)
    metrics = compute_mt_metrics(predictions, references)

    assert isinstance(bleu, float)
    assert isinstance(chrf, float)
    assert isinstance(rouge, dict)
    assert isinstance(metrics, dict)

    assert "eval/bleu" in metrics
    assert "eval/chrf" in metrics
    assert "eval/rouge1" in metrics
    assert "eval/rouge2" in metrics
    assert "eval/rougeL" in metrics

    print("metric functions passed")

except ImportError as e:
    print("metric functions skipped because dependency is missing")
    print(e)

metric functions passed


In [15]:
# test evaluate model with tuple dataloader

class TinyEvalModel(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.proj = torch.nn.Linear(4, vocab_size)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        x = torch.ones(batch_size, seq_len, 4, device=input_ids.device)
        return self.proj(x)

vocab_size = 6
model = TinyEvalModel(vocab_size)

inputs = torch.randint(0, vocab_size, (3, 5))
labels = torch.randint(0, vocab_size, (3, 5))

eval_loader = DataLoader(
    TensorDataset(inputs, labels),
    batch_size=2,
)

try:
    metrics = evaluate_model(
        model=model,
        eval_dataloader=eval_loader,
        tokenizer=None,
        device="cpu",
    )

    assert isinstance(metrics, dict)
    assert "eval/bleu" in metrics
    assert "eval/chrf" in metrics

    print("evaluate model tuple dataloader passed")

except ImportError as e:
    print("evaluate model skipped because metric dependency is missing")
    print(e)

evaluate model tuple dataloader passed


In [16]:
# test evaluate model with dict dataloader and generate

class TinyGenerateModel(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.dummy = torch.nn.Parameter(torch.zeros(1))
        self.vocab_size = vocab_size

    def generate(self, input_ids, **kwargs):
        return input_ids.clone()

vocab_size = 8
model = TinyGenerateModel(vocab_size)

dict_batches = [
    {
        "input_ids": torch.tensor([[1, 2, 3], [4, 5, 6]]),
        "labels": torch.tensor([[1, 2, 3], [4, 5, 6]]),
    }
]

try:
    metrics = evaluate_model(
        model=model,
        eval_dataloader=dict_batches,
        tokenizer=None,
        device="cpu",
    )

    assert isinstance(metrics, dict)
    assert "eval/bleu" in metrics

    print("evaluate model dict dataloader passed")

except ImportError as e:
    print("evaluate model skipped because metric dependency is missing")
    print(e)

evaluate model dict dataloader passed


In [17]:
# test evaluate model with dict dataloader and generate

class TinyGenerateModel(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.dummy = torch.nn.Parameter(torch.zeros(1))
        self.vocab_size = vocab_size

    def generate(self, input_ids, **kwargs):
        return input_ids.clone()

vocab_size = 8
model = TinyGenerateModel(vocab_size)

dict_batches = [
    {
        "input_ids": torch.tensor([[1, 2, 3], [4, 5, 6]]),
        "labels": torch.tensor([[1, 2, 3], [4, 5, 6]]),
    }
]

try:
    metrics = evaluate_model(
        model=model,
        eval_dataloader=dict_batches,
        tokenizer=None,
        device="cpu",
    )

    assert isinstance(metrics, dict)
    assert "eval/bleu" in metrics

    print("evaluate model dict dataloader passed")

except ImportError as e:
    print("evaluate model skipped because metric dependency is missing")
    print(e)

evaluate model dict dataloader passed


In [18]:
# test trainer helpers

batch = {
    "input_ids": torch.tensor([[1, 2], [3, 4]]),
    "labels": torch.tensor([[1, 2], [3, 4]]),
}

moved = _move_to_device(batch, torch.device("cpu"))
labels = _extract_labels(moved)

assert isinstance(moved, dict)
assert labels.shape == (2, 2)

loader = [batch, batch, batch, batch]

steps = _compute_total_training_steps(
    train_dataloader=loader,
    num_epochs=2,
    grad_accum_steps=2,
    max_steps=None,
)

assert steps == 4

steps = _compute_total_training_steps(
    train_dataloader=loader,
    num_epochs=2,
    grad_accum_steps=2,
    max_steps=3,
)

assert steps == 3

print("trainer helpers passed")

trainer helpers passed


In [19]:
# test train minimal

class TinyTrainModel(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, 8)
        self.proj = torch.nn.Linear(8, vocab_size)

    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        logits = self.proj(x)
        return {"logits": logits}

vocab_size = 12
seq_len = 5

input_ids = torch.randint(0, vocab_size, (8, seq_len))
labels = torch.randint(0, vocab_size, (8, seq_len))

train_loader = DataLoader(
    TensorDataset(input_ids, labels),
    batch_size=2,
)

model = TinyTrainModel(vocab_size)

config = {
    "output_dir": str(PROJECT_ROOT / "tmp_test_checkpoints"),
    "num_epochs": 1,
    "max_steps": 2,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "group_weight_decay": False,
    "scheduler_type": "none",
    "logging_steps": 1,
    "save_steps": None,
    "run_eval_at_end": False,
    "wandb_enabled": False,
    "mixed_precision": False,
    "label_smoothing": 0.0,
    "ignore_index": -100,
}

result = train(
    model=model,
    train_dataloader=train_loader,
    eval_dataloader=None,
    config=config,
    tokenizer=None,
    device="cpu",
)

assert isinstance(result, dict)
assert result["global_step"] == 2
assert result["last_checkpoint"] is not None

last_checkpoint = Path(result["last_checkpoint"])

assert last_checkpoint.exists()
assert (last_checkpoint / "model.pt").exists()
assert (last_checkpoint / "training_state.pt").exists()

print("train minimal passed")

train minimal passed


/home/lqb464/Desktop/machine-translation-from-scratch/src/training/trainer.py:252: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/lqb464/Desktop/machine-translation-from-scratch/src/training/trainer.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


In [20]:
# test train with eval disabled at end but checkpoint exists

class TinyTrainModelWithGenerate(torch.nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, 8)
        self.proj = torch.nn.Linear(8, vocab_size)

    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        logits = self.proj(x)
        return {"logits": logits}

    def generate(self, input_ids, **kwargs):
        return input_ids.clone()

vocab_size = 10
seq_len = 4

input_ids = torch.randint(0, vocab_size, (4, seq_len))
labels = torch.randint(0, vocab_size, (4, seq_len))

train_loader = DataLoader(
    TensorDataset(input_ids, labels),
    batch_size=2,
)

model = TinyTrainModelWithGenerate(vocab_size)

config = {
    "output_dir": str(PROJECT_ROOT / "tmp_test_checkpoints_eval"),
    "num_epochs": 1,
    "max_steps": 1,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "group_weight_decay": False,
    "scheduler_type": "linear",
    "warmup_steps": 0,
    "logging_steps": 1,
    "save_steps": 1,
    "run_eval_at_end": False,
    "wandb_enabled": False,
    "mixed_precision": False,
}

result = train(
    model=model,
    train_dataloader=train_loader,
    eval_dataloader=None,
    config=config,
    tokenizer=None,
    device="cpu",
)

assert result["global_step"] == 1
assert Path(result["last_checkpoint"]).exists()
assert (PROJECT_ROOT / "tmp_test_checkpoints_eval" / "step_1").exists()

print("train checkpoint passed")

train checkpoint passed
